# NSL-KDD - Análise exploratória e diagnóstico inicial

Entrega da Etapa 1: usa exclusivamente `treino.csv`, sem treinamento ou decisão definitiva de pré-processamento.

## 1. Carregamento reprodutível

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
RAIZ=Path.cwd()
if not (RAIZ / "preprocessamento.py").is_file(): raise FileNotFoundError("Abra na raiz do projeto.")
sys.path.insert(0,str(RAIZ))
from preprocessamento import *
DATA_PATH=RAIZ / "treino.csv"
dados=carregar_treino(DATA_PATH)
print(DATA_PATH.name, dados.shape)

## 2. Caracterização e desbalanceamento

In [ ]:
numericas,categoricas=identificar_colunas_por_tipo(dados)
perfil=pd.DataFrame({"tipo":dados.dtypes.astype(str),"ausencias":dados.isna().sum(),"distintos":dados.nunique(dropna=False)})
display(perfil)
classe=dados[ALVO].value_counts().rename("frequencia").to_frame(); classe["percentual"]=(100*classe.frequencia/len(dados)).round(3)
display(classe)
sns.countplot(data=dados,x=ALVO,order=classe.index,hue=ALVO,legend=False); plt.title("Distribuição de class"); plt.show()

A predominância de `normal` caracteriza desbalanceamento. Nenhum balanceamento é aplicado agora; ele será comparado em validação na Etapa 2/3.

## 3. Investigação do atributo `su_attempted`

In [ ]:
su=investigar_su_attempted(dados)
display(su["frequencias"]); display(su["por_classe"]); display(su["por_servico"]); display(su["por_root_shell"])
print("Registros com valor 2:",len(su["registros_com_valor_2"])); print("Duplicatas nesse subconjunto:",su["duplicatas_valor_2"])

Pela definição original, `su_attempted` é binário; portanto, `2` é inconsistente. Não será recodificado. Na Etapa 2 serão comparadas: categoria especial, ausência com imputação somente no treino, ou exclusão apenas dos registros afetados.

## 4. Regras de domínio e exemplos

In [ ]:
print("Taxas [0,1]:",identificar_taxas(dados))
display(analisar_violacoes_dominio(dados))
resumo_taxas=resumir_taxas_invalidas(dados)
print("Registros com ao menos uma taxa inválida:",resumo_taxas["registros_com_taxa_invalida"], f"({resumo_taxas['percentual_base']}% da base)")
display(exemplos_violacoes(dados))

## 5. Caixa e extremos potencialmente legítimos

In [ ]:
display(encontrar_inconsistencias_caixa(dados,categoricas))
interesse=[x for x in ["duration","src_bytes","dst_bytes","num_root","num_compromised"] if x in dados]
display(dados[interesse].describe(percentiles=[.01,.05,.5,.95,.99]).T)
for coluna in interesse:
 validos=dados.loc[dados[coluna].ge(0),coluna].dropna()
 fig,ax=plt.subplots(1,2,figsize=(12,3)); sns.boxplot(x=dados[coluna],ax=ax[0]); ax[0].set_title(coluna); sns.histplot(np.log1p(validos),bins=40,ax=ax[1]); ax[1].set_title(f"log1p({coluna}) - apenas valores válidos"); plt.show()

Nem todo extremo é erro: evento raro pode ser relevante para intrusão. Nenhum outlier é removido nesta etapa. Para a visualização logarítmica, somente valores não negativos são exibidos; essa filtragem é exclusivamente gráfica e não altera os dados originais.

## 6. Redundâncias, duplicatas e conflitos

In [ ]:
display(verificar_redundancias(dados))
resumo,conflitos=analisar_duplicatas(dados); display(resumo); display(conflitos.head())

## 7. Espaço reservado para investigação de leakage

In [ ]:
print("Sob investigação específica, sem conclusão aqui:",[x for x in COLUNAS_SUSPEITAS_LEAKAGE if x in dados])

# Diagnóstico da análise exploratória

A síntese abaixo é calculada a partir da base carregada nesta execução.

In [ ]:
resumo_classes=dados[ALVO].value_counts(normalize=True).mul(100).round(2)
print(f"Desbalanceamento: normal={resumo_classes.get('normal',0)}% e anomaly={resumo_classes.get('anomaly',0)}%. Por isso, acurácia isolada será inadequada nas próximas etapas.")
print("Inconsistências confirmadas:", analisar_violacoes_dominio(dados).query("violacoes > 0")[["atributo","violacoes"]].to_dict("records"))
print("Inconsistências categóricas:", encontrar_inconsistencias_caixa(dados,categoricas).to_dict("records"))
print("Redundâncias comprovadas:", verificar_redundancias(dados).query("redundancia_comprovada")["atributo"].tolist())
print("Extremos positivos permanecem sem remoção automática, pois podem representar eventos raros genuínos.")
print("Etapa 2: comparar tratamento de inconsistências, su_attempted, redundâncias, imputação, codificação, transformação, outliers e balanceamento dentro do treino.")